<a href="https://colab.research.google.com/github/rafi1624/Machine-Learning_18_Rafi-Abyantara/blob/main/JS09/JS09%20-%20Tugas%20Lab%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install annoy faiss-cpu hnswlib scikit-learn pandas numpy kagglehub -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import time
import numpy as np
import pandas as pd
import kagglehub
import os
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from annoy import AnnoyIndex
import faiss
import hnswlib

# ==========================================
# 1. Download & Load Dataset Spotify
# ==========================================
print("Mengunduh dataset dari Kaggle...")
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
csv_files.sort() # Pilih file pertama dari dua dataset yang tersedia

csv_path = os.path.join(path, csv_files[0])
print(f"Memuat file: {csv_files[0]}")
df = pd.read_csv(csv_path)

# ==========================================
# 2. Preprocessing & Normalisasi Fitur Numerik
# ==========================================
features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

# Hapus baris dengan nilai missing pada fitur numerik
df_clean = df.dropna(subset=features).reset_index(drop=True)

X = df_clean[features].values.astype(np.float32)
dim = X.shape[1]
n_samples = X.shape[0]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

k = 10  # Jumlah lagu terdekat yang dicari
query_idx = 0  # Menggunakan lagu pertama sebagai sampel query
query_vector = X_scaled[query_idx:query_idx+1]

print(f"\nTotal Data: {n_samples} lagu | Jumlah Fitur: {dim}D")
if 'track_name' in df_clean.columns:
    print(f"Lagu Query: '{df_clean.loc[query_idx, 'track_name']}' oleh {df_clean.loc[query_idx, 'track_artist']}\n")

# ==========================================
# 3. Benchmark Algoritma
# ==========================================

# --- A. Exact Nearest Neighbors (Scikit-Learn - Ground Truth) ---
start = time.time()
nn_exact = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn_exact.fit(X_scaled)
exact_build_time = time.time() - start

start = time.time()
exact_distances, exact_indices = nn_exact.kneighbors(query_vector)
exact_query_time = time.time() - start
ground_truth = set(exact_indices[0])

# --- B. ANNOY ---
ann_index = AnnoyIndex(dim, 'euclidean')
start = time.time()
for i in range(n_samples):
    ann_index.add_item(i, X_scaled[i])
ann_index.build(10)  # 10 trees
annoy_build_time = time.time() - start

start = time.time()
annoy_indices = ann_index.get_nns_by_vector(query_vector[0], k)
annoy_query_time = time.time() - start
annoy_recall = len(ground_truth.intersection(set(annoy_indices))) / k

# --- C. FAISS (IndexFlatL2) ---
faiss_index = faiss.IndexFlatL2(dim)
start = time.time()
faiss_index.add(X_scaled)
faiss_build_time = time.time() - start

start = time.time()
faiss_distances, faiss_indices = faiss_index.search(query_vector, k)
faiss_query_time = time.time() - start
faiss_recall = len(ground_truth.intersection(set(faiss_indices[0]))) / k

# --- D. HNSW (hnswlib) ---
hnsw_index = hnswlib.Index(space='l2', dim=dim)
start = time.time()
hnsw_index.init_index(max_elements=n_samples, ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
hnsw_build_time = time.time() - start

hnsw_index.set_ef(50)
start = time.time()
hnsw_labels, hnsw_distances = hnsw_index.knn_query(query_vector, k=k)
hnsw_query_time = time.time() - start
hnsw_recall = len(ground_truth.intersection(set(hnsw_labels[0]))) / k

# ==========================================
# 4. Cetak Tabel Markdown Hasil Eksperimen
# ==========================================
md_table = "| Algoritma | Waktu Build (s) | Waktu Query (s) | Recall@10 (vs Exact NN) |\n"
md_table += "| :--- | :---: | :---: | :---: |\n"
md_table += f"| **Exact NN (Brute)** | {exact_build_time:.4f}s | {exact_query_time:.5f}s | 100% |\n"
md_table += f"| **ANNOY** | {annoy_build_time:.4f}s | {annoy_query_time:.5f}s | {annoy_recall*100:.1f}% |\n"
md_table += f"| **FAISS (FlatL2)** | {faiss_build_time:.4f}s | {faiss_query_time:.5f}s | {faiss_recall*100:.1f}% |\n"
md_table += f"| **HNSW** | {hnsw_build_time:.4f}s | {hnsw_query_time:.5f}s | {hnsw_recall*100:.1f}% |\n"

print("="*70)
print("HASIL BENCHMARK TEKS MARKDOWN (Siap paste ke Cell + Teks Colab):")
print("="*70 + "\n")
print(md_table)

Mengunduh dataset dari Kaggle...
Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Memuat file: songs_with_attributes_and_lyrics.csv

Total Data: 955320 lagu | Jumlah Fitur: 9D
HASIL BENCHMARK TEKS MARKDOWN (Siap paste ke Cell + Teks Colab):

| Algoritma | Waktu Build (s) | Waktu Query (s) | Recall@10 (vs Exact NN) |
| :--- | :---: | :---: | :---: |
| **Exact NN (Brute)** | 0.0303s | 0.34822s | 100% |
| **ANNOY** | 16.1831s | 0.00020s | 60.0% |
| **FAISS (FlatL2)** | 0.0427s | 0.01077s | 100.0% |
| **HNSW** | 168.0988s | 0.00097s | 100.0% |

